# 07 Limitations And Data Budget

This notebook answers the final practical question: how much headroom is likely left, and where does the current thesis package still remain scientifically limited?

**Questions answered here**
- Are we still on the steep part of the paired-label curve?
- How much performance is lost under family-unseen transfer?
- Which limitations are methodological, and which are data-limited?


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def _find_thesis_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    nested = [candidate / "thesis" for candidate in direct]
    for candidate in [*direct, *nested]:
        if (
            (candidate / "src" / "qc_thesis" / "__init__.py").exists()
            and (candidate / "README.md").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not find thesis root from notebook session")

ROOT = _find_thesis_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()


def show_saved_figure(path, caption=None):
    figure_path = Path(path)
    if not figure_path.is_absolute():
        figure_path = (ROOT / figure_path).resolve()
    if not figure_path.exists():
        display(Markdown(f"_Missing figure: `{figure_path}`_"))
        return
    if caption:
        display(Markdown(caption))
    display(Image(filename=str(figure_path)))

fig_dir, table_dir = notebook_output_dirs("07_limitations_and_data_budget")
bundle = get_limitations_bundle()
fpos_progress = build_benchmark_progress_table("fpos", scope="core")
fmiss_progress = build_benchmark_progress_table("fmiss", scope="core")


## 1. Label-budget evidence

The thesis should make the small-data regime visible. If the curve is still rising steeply, then the current benchmark is probably data-limited rather than fully saturated.


In [ ]:
display(bundle.label_budget)
save_table(bundle.label_budget, table_dir, "label_budget_summary")
fig, _ = plot_label_budget(bundle.label_budget, target="fpos")
save_figure(fig, fig_dir, "label_budget_fpos")
fig


In [ ]:
fig, _ = plot_label_budget(bundle.label_budget, target="fmiss")
save_figure(fig, fig_dir, "label_budget_fmiss")
fig


In [ ]:
show_saved_figure(
    fig_dir / "label_budget_combined.png",
    "The combined label-budget figure is the cleanest way to show both targets in the same small-data frame instead of reading two separate panels in isolation.",
)


## 2. Family-held-out robustness

The main benchmark optimizes recording-disjoint transfer. This section reports the explicit penalty for requiring generalization to an unseen paired family.


In [ ]:
display(bundle.lofo_aggregate)
display(bundle.lofo_family)
save_table(bundle.lofo_aggregate, table_dir, "leave_one_family_aggregate")
save_table(bundle.lofo_family, table_dir, "leave_one_family_by_family")
fig, _, _ = plot_group_metric(bundle.lofo_family, group_col="held_out_family", metric="r2", title="Leave-one-family-out `fpos` R²")
save_figure(fig, fig_dir, "leave_one_family_r2")
fig


## 3. What the current results imply about headroom

The table below puts the current best results next to the dummy floor. It is not a theoretical upper bound, but it does show how much usable signal the retained models recover above a non-informative predictor.


In [ ]:
basic_fpos = fpos_progress.loc[fpos_progress["recipe_id"] == "fpos_dummy"].iloc[0]
final_fpos = fpos_progress.sort_values("r2", ascending=False).iloc[0]
basic_fmiss = fmiss_progress.loc[fmiss_progress["recipe_id"] == "fmiss_dummy"].iloc[0]
final_fmiss = fmiss_progress.sort_values("r2", ascending=False).iloc[0]
headroom = pd.DataFrame([
    {"target": "fpos", "basic_transfer_r2": basic_fpos["r2"], "current_best_r2": final_fpos["r2"], "gain_r2": final_fpos["r2"] - basic_fpos["r2"]},
    {"target": "fmiss", "basic_transfer_r2": basic_fmiss["r2"], "current_best_r2": final_fmiss["r2"], "gain_r2": final_fmiss["r2"] - basic_fmiss["r2"]},
])
display(headroom)
save_table(headroom, table_dir, "headroom_summary")


In [ ]:
show_saved_figure(
    fig_dir / "headroom_summary.png",
    "This headroom summary shows how far each target moved from the dummy floor to the current best retained model, which is more interpretable than a raw bar chart of the final rows alone.",
)


In [ ]:
display(Markdown(
    """
## Limitation summary

- The paired labeled pool is still small enough that label-budget curves matter.
- Leave-one-family-out performance remains clearly worse than the main benchmark, so family-unseen robustness is not solved.
- The strongest positive result is currently clearer for `fpos` than for `fmiss`.
- The most plausible next gains should come from richer spatial/template/drift information, not from more generic backend tinkering.
"""
))
